# 02 — Max-Cut and the Quantum Approximate Optimization Algorithm

**Phase 2 | Qiskit optimization baseline**

Build a graph optimization objective, establish exact and heuristic classical baselines, implement QAOA circuits from first principles, and benchmark simulated solution quality. All calculations are self-contained in this notebook. **These are small, ideal-simulator experiments—not evidence of quantum advantage.**

### 1. Load libraries and configure the experiment

Use fixed seeds for graph layouts, optimizer starts, and quantum measurement sampling. Write experiment results to the existing `reports` directories.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

SEED = 31
SHOTS = 8192
DEPTHS = (1, 2)
RESTARTS = 3
rng = np.random.default_rng(SEED)

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "notebooks").exists(), f"Run this notebook inside the QuantumOptLab project: {ROOT}"
EXPERIMENTS = ROOT / "reports" / "experiments"
FIGURES = ROOT / "reports" / "figures"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Qiskit:", qiskit.__version__, "| Aer:", qiskit_aer.__version__)
print("Project:", ROOT)
print("Seed:", SEED, "| Shots per final circuit:", SHOTS)

### 2. Create three small Max-Cut instances

Use an unweighted cycle and two weighted, connected graphs. The edge weights are positive integers so that the angle search ranges are well-defined for this initial implementation.

In [ ]:
GRAPH_EDGES = {
    "cycle_4": [
        (0, 1, 1), (1, 2, 1), (2, 3, 1), (3, 0, 1)
    ],
    "weighted_5": [
        (0, 1, 1), (0, 2, 2), (1, 2, 1), (1, 3, 1),
        (2, 3, 2), (2, 4, 1), (3, 4, 1)
    ],
    "weighted_6": [
        (0, 1, 1), (1, 2, 2), (2, 3, 1), (3, 4, 2),
        (4, 5, 1), (5, 0, 2), (0, 2, 1), (1, 4, 1), (3, 5, 1)
    ],
}
GRAPHS = {}
for name, edges in GRAPH_EDGES.items():
    graph = nx.Graph()
    graph.add_weighted_edges_from(edges)
    graph = nx.convert_node_labels_to_integers(graph, ordering="sorted")
    assert nx.is_connected(graph)
    GRAPHS[name] = graph

fig, axes = plt.subplots(1, len(GRAPHS), figsize=(12, 3.3))
for ax, (name, graph) in zip(axes, GRAPHS.items()):
    pos = nx.spring_layout(graph, seed=SEED)
    nx.draw_networkx(graph, pos=pos, ax=ax, node_size=570, font_size=10)
    edge_labels = nx.get_edge_attributes(graph, "weight")
    nx.draw_networkx_edge_labels(graph, pos=pos, edge_labels={k:int(v) for k,v in edge_labels.items()}, ax=ax, font_size=8)
    ax.set_title(f"{name} | {graph.number_of_nodes()} nodes")
    ax.axis("off")
fig.tight_layout()
graph_path = FIGURES / "phase02_graph_instances_seed31.png"
fig.savefig(graph_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", graph_path)

### 3. Calculate exact Max-Cut values and classical baselines

For a bit assignment $x\in\{0,1\}^{n}$, maximize

$$C(x)=\sum_{(i,j)\in E} w_{ij}(x_i+x_j-2x_ix_j).$$

Enumerate every bitstring to obtain the exact optimum on these small graphs. Also compute the uniform-random expectation and a greedy bit-flip local-search baseline with multiple random starts. Qiskit bitstrings are displayed with qubit 0 on the **right**.

In [ ]:
def cut_value(graph, assignment):
    """Weighted cut of assignment encoded as a nonnegative integer."""
    return float(sum(
        data["weight"] for u, v, data in graph.edges(data=True)
        if ((assignment >> u) & 1) != ((assignment >> v) & 1)
    ))


def exact_solution(graph):
    values = np.array([cut_value(graph, k) for k in range(1 << graph.number_of_nodes())], dtype=float)
    optimum = float(values.max())
    states = np.flatnonzero(np.isclose(values, optimum)).tolist()
    return values, optimum, states


def greedy_local_search(graph, seed=SEED, n_starts=20):
    """A classical heuristic; separate evaluation budget from QAOA."""
    generator = np.random.default_rng(seed)
    n = graph.number_of_nodes()
    best_assignment, best_value = 0, -np.inf
    for _ in range(n_starts):
        x = int(generator.integers(0, 1 << n))
        while True:
            neighbors = [x ^ (1 << j) for j in range(n)]
            candidates = [cut_value(graph, trial) for trial in neighbors]
            j = int(np.argmax(candidates))
            if candidates[j] > cut_value(graph, x) + 1e-12:
                x = neighbors[j]
            else:
                break
        value = cut_value(graph, x)
        if value > best_value:
            best_value, best_assignment = value, x
    return best_assignment, best_value


BASELINES = {}
for name, graph in GRAPHS.items():
    costs, optimum, opt_states = exact_solution(graph)
    greedy_state, greedy_value = greedy_local_search(graph)
    BASELINES[name] = dict(
        costs=costs, optimum=optimum, optimal_states=opt_states,
        uniform_mean=float(costs.mean()), greedy_value=float(greedy_value),
        greedy_state=int(greedy_state)
    )

baseline_df = pd.DataFrame([{
    "graph": name, "qubits": graph.number_of_nodes(),
    "edges": graph.number_of_edges(), "uniform_mean": data["uniform_mean"],
    "greedy_cut": data["greedy_value"], "exact_optimum": data["optimum"],
    "n_optimal_bitstrings": len(data["optimal_states"])
} for name, graph in GRAPHS.items() for data in [BASELINES[name]]])
display(baseline_df.round(4))

### 4. Map the graph objective to an Ising Hamiltonian

Use $x_i=(1-Z_i)/2$ to form

$$H_C=\sum_{(i,j)\in E}\frac{w_{ij}}{2}(I-Z_iZ_j).$$

The computational-basis eigenvalue of $H_C$ must equal the classical cut value. Verify **all** basis states, rather than relying on a single example.

In [ ]:
def maxcut_hamiltonian(graph):
    n = graph.number_of_nodes()
    edges = [(u, v, float(data["weight"])) for u, v, data in graph.edges(data=True)]
    constant = sum(w for _, _, w in edges) / 2.0
    h_identity = SparsePauliOp.from_list([("I" * n, constant)])
    h_zz = SparsePauliOp.from_sparse_list(
        [("ZZ", [u, v], -w / 2.0) for u, v, w in edges], num_qubits=n
    )
    return (h_identity + h_zz).simplify()

HAMILTONIANS = {}
for name, graph in GRAPHS.items():
    hamiltonian = maxcut_hamiltonian(graph)
    HAMILTONIANS[name] = hamiltonian
    eigenvalues = np.diag(hamiltonian.to_matrix()).real
    mismatch = float(np.max(np.abs(eigenvalues - BASELINES[name]["costs"])))
    assert mismatch < 1e-10, (name, mismatch)
    print(f"{name}: Hamiltonian matches all {len(eigenvalues)} classical assignments; max error={mismatch:.2e}")

### 5. Implement QAOA with Qiskit gates

Start each qubit in $|+\rangle$. For layer $\ell$, apply cost evolution $e^{-i\gamma_\ell H_C}$ and mixer evolution $e^{-i\beta_\ell\sum_i X_i}$. The cost unitary is implemented with `rzz(-gamma * weight)` up to a global phase; the mixer uses `rx(2 * beta)`.

Optimize the **expected cut value** from the exact statevector. This deliberately removes finite-shot fluctuations from the first optimizer comparison.

In [ ]:
def qaoa_circuit(graph, params, depth):
    n = graph.number_of_nodes()
    params = np.asarray(params, dtype=float)
    assert params.shape == (2 * depth,)
    gammas, betas = params[:depth], params[depth:]
    circuit = QuantumCircuit(n)
    circuit.h(range(n))
    for gamma, beta in zip(gammas, betas):
        for u, v, data in graph.edges(data=True):
            circuit.rzz(-float(gamma) * float(data["weight"]), u, v)
        for q in range(n):
            circuit.rx(2.0 * float(beta), q)
    return circuit


def statevector_stats(graph, params, depth, costs, optimum):
    state = Statevector.from_instruction(qaoa_circuit(graph, params, depth))
    probs = np.abs(state.data)**2
    mean_cut = float(np.dot(probs, costs))
    optimal_probability = float(probs[np.isclose(costs, optimum)].sum())
    return mean_cut, optimal_probability, probs


example = qaoa_circuit(GRAPHS["cycle_4"], [0.60, 0.35], depth=1)
print("Example QAOA p=1 circuit:")
print(example.draw(output="text", fold=120))
print("Depth:", example.depth(), "| Operations:", dict(example.count_ops()))

# If gamma is zero, cost evolution is identity; with mixer only,
# the initial uniform state is unchanged up to phase.
for name, graph in GRAPHS.items():
    data = BASELINES[name]
    mean_cut, _, probs = statevector_stats(
        graph, params=[0.0, 0.4], depth=1,
        costs=data["costs"], optimum=data["optimum"]
    )
    assert np.allclose(probs, np.ones(1 << graph.number_of_nodes()) / (1 << graph.number_of_nodes()), atol=1e-10)
    assert np.isclose(mean_cut, data["uniform_mean"], atol=1e-10)
print("PASS: zero-cost-angle QAOA gives the expected uniform distribution on every graph")

### 6. Optimize QAOA parameters at depths p = 1 and p = 2

Perform three reproducible restarts per graph/depth using SciPy L-BFGS-B. Restrict gamma to `[0, 2π]` and beta to `[0, π]` (positive integer edge weights). Record optimizer objective evaluations. Retain the best restart based on expected cut, not on a sampled maximum.

In [ ]:
TRAINED = {}
optimization_rows = []
start_time = time.perf_counter()

for graph_index, (name, graph) in enumerate(GRAPHS.items()):
    data = BASELINES[name]
    for depth in DEPTHS:
        best = None
        total_calls = 0
        for restart in range(RESTARTS):
            start_rng = np.random.default_rng(SEED + 1000 * graph_index + 100 * depth + restart)
            initial = np.concatenate([
                start_rng.uniform(0, 2*np.pi, size=depth),
                start_rng.uniform(0, np.pi, size=depth)
            ])
            calls = [0]
            trajectory = []

            def objective(params):
                calls[0] += 1
                expected, _, _ = statevector_stats(graph, params, depth, data["costs"], data["optimum"])
                trajectory.append(expected)
                return -expected

            outcome = minimize(
                objective, initial, method="L-BFGS-B",
                bounds=[(0.0, 2*np.pi)]*depth + [(0.0, np.pi)]*depth,
                options={"maxiter": 90, "maxfun": 450, "ftol": 1e-10}
            )
            expectation, probability, probs = statevector_stats(
                graph, outcome.x, depth, data["costs"], data["optimum"]
            )
            total_calls += calls[0]
            candidate = {
                "params": np.asarray(outcome.x).copy(), "expected": expectation,
                "p_optimal": probability, "probs": probs,
                "best_restart": restart, "best_restart_calls": calls[0],
                "best_trajectory": trajectory.copy(), "optimizer_success": bool(outcome.success),
                "optimizer_message": str(outcome.message)
            }
            if best is None or candidate["expected"] > best["expected"]:
                best = candidate
        best["total_objective_calls"] = total_calls
        TRAINED[(name, depth)] = best
        optimization_rows.append({
            "graph": name, "qubits": graph.number_of_nodes(), "depth_p": depth,
            "uniform_mean": data["uniform_mean"], "greedy_cut": data["greedy_value"],
            "exact_optimum": data["optimum"], "qaoa_expected_cut": best["expected"],
            "approximation_ratio": best["expected"] / data["optimum"],
            "probability_optimal_bitstring": best["p_optimal"],
            "best_restart": best["best_restart"],
            "optimizer_success": best["optimizer_success"],
            "optimizer_objective_calls": total_calls
        })
        print(f"{name:12s} p={depth}: E[cut]={best['expected']:.4f} / optimum={data['optimum']:.1f}; P(optimum)={best['p_optimal']:.3f}; evals={total_calls}")

optimization_df = pd.DataFrame(optimization_rows)
print(f"Optimization elapsed: {time.perf_counter()-start_time:.1f}s")
display(optimization_df.round(4))

### 7. Verify trained circuits with finite-shot simulation

Measure the best circuit using 8,192 ideal shots. Compare the shot-estimated expected cut with the exact statevector expectation. Track the frequency of sampling a globally optimal bitstring; the best observed cut from thousands of samples is not, by itself, a fair algorithm-quality metric.

In [ ]:
simulator = AerSimulator()
sampling_rows = []
for index, ((name, depth), run) in enumerate(TRAINED.items()):
    graph = GRAPHS[name]
    data = BASELINES[name]
    circuit = qaoa_circuit(graph, run["params"], depth)
    circuit.measure_all()
    result = simulator.run(circuit, shots=SHOTS, seed_simulator=SEED + index).result()
    counts = result.get_counts()
    assert sum(counts.values()) == SHOTS
    estimated_mean = sum(cut_value(graph, int(bits.replace(" ", ""), 2)) * count for bits, count in counts.items()) / SHOTS
    estimated_optimal_prob = sum(count for bits, count in counts.items()
        if np.isclose(cut_value(graph, int(bits.replace(" ", ""), 2)), data["optimum"])) / SHOTS
    sampling_rows.append({
        "graph": name, "depth_p": depth, "shots": SHOTS,
        "statevector_expected_cut": run["expected"], "sampled_expected_cut": estimated_mean,
        "absolute_mean_difference": abs(estimated_mean - run["expected"]),
        "statevector_p_optimal": run["p_optimal"],
        "sampled_p_optimal": estimated_optimal_prob,
        "circuit_depth_untranspiled": circuit.depth(),
        "rzz_count_untranspiled": int(circuit.count_ops().get("rzz", 0)),
    })

sampling_df = pd.DataFrame(sampling_rows)
display(sampling_df.round(4))

### 8. Compare quantum and classical results visually

Plot expected QAOA cut against the exact optimum and classical baselines. Keep the optimal-bitstring probability as a separate metric so that it is not confused with the approximation ratio.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
names = list(GRAPHS.keys())
xx = np.arange(len(names))
width = 0.20
for offset, depth in [(-0.5, 1), (0.5, 2)]:
    sub = optimization_df.set_index(["graph", "depth_p"])
    axes[0].bar(xx + offset*width, [sub.loc[(name, depth), "qaoa_expected_cut"] for name in names],
                width=width, label=f"QAOA p={depth}")
axes[0].bar(xx - 1.5*width, [BASELINES[name]["uniform_mean"] for name in names], width=width, label="Uniform random")
axes[0].bar(xx + 1.5*width, [BASELINES[name]["greedy_value"] for name in names], width=width, label="Greedy local search")
axes[0].plot(xx, [BASELINES[name]["optimum"] for name in names], "k_", markersize=18, markeredgewidth=2, label="Exact optimum")
axes[0].set_ylabel("Cut value (expectation for QAOA)")
axes[0].set_xticks(xx, names, rotation=15)
axes[0].set_title("Optimization quality")
axes[0].legend(fontsize=8)

for depth in DEPTHS:
    sub = optimization_df[optimization_df.depth_p == depth].set_index("graph")
    axes[1].plot(names, [sub.loc[name, "probability_optimal_bitstring"] for name in names],
                 "o-", label=f"p={depth}")
axes[1].set_ylim(0, 1.02)
axes[1].set_ylabel("Probability of an optimal bitstring")
axes[1].set_title("Ideal-state success probability")
axes[1].legend()
fig.tight_layout()
comparison_path = FIGURES / "phase02_maxcut_qaoa_comparison_seed31.png"
fig.savefig(comparison_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", comparison_path)

### 9. Save benchmark results and verify invariants

Write CSV tables and a JSON manifest containing the seed, optimizer budgets, and best circuit parameters. Check classical-to-Hamiltonian equivalence, normalized quantum probabilities, valid optimization bounds, and shot counts.

In [ ]:
checks = {
    "three connected graphs": len(GRAPHS) == 3 and all(nx.is_connected(g) for g in GRAPHS.values()),
    "exact cut is globally maximal": all(np.isclose(b["optimum"], max(b["costs"])) for b in BASELINES.values()),
    "cut complement symmetry": all(all(np.isclose(cut_value(g, k), cut_value(g, k ^ ((1 << g.number_of_nodes()) - 1))) for k in range(1 << g.number_of_nodes())) for g in GRAPHS.values()),
    "Hamiltonian matches all assignments": all(np.allclose(np.diag(HAMILTONIANS[name].to_matrix()).real, BASELINES[name]["costs"]) for name in GRAPHS),
    "all six QAOA runs completed": len(TRAINED) == len(GRAPHS)*len(DEPTHS),
    "QAOA probability vectors normalized": all(np.isclose(np.sum(run["probs"]), 1.0, atol=1e-10) for run in TRAINED.values()),
    "QAOA expectations in exact bounds": all(-1e-8 <= run["expected"] <= BASELINES[name]["optimum"] + 1e-8 for (name, depth), run in TRAINED.items()),
    "finite-shot results recorded": len(sampling_df) == len(TRAINED) and bool((sampling_df["shots"] == SHOTS).all()),
}
for label, passed in checks.items():
    print(("PASS" if passed else "FAIL") + ": " + label)
assert all(checks.values()), "At least one Phase 2 check failed"
print(f"PASS: {sum(checks.values())}/{len(checks)} Phase 2 checks")

optimization_path = EXPERIMENTS / "phase02_qaoa_optimization_seed31.csv"
sampling_path = EXPERIMENTS / "phase02_qaoa_shot_validation_seed31.csv"
manifest_path = EXPERIMENTS / "phase02_manifest_seed31.json"
optimization_df.to_csv(optimization_path, index=False)
sampling_df.to_csv(sampling_path, index=False)
manifest = {
    "phase": 2, "seed": SEED, "shots_per_final_circuit": SHOTS,
    "algorithm": "statevector-optimized QAOA",
    "depths": list(DEPTHS), "optimizer": "SciPy L-BFGS-B",
    "restarts_per_graph_depth": RESTARTS, "optimizer_maxiter": 90,
    "optimizer_maxfun": 450,
    "graph_edges": {name: [[int(u), int(v), int(w)] for u, v, w in edges] for name, edges in GRAPH_EDGES.items()},
    "best_parameters": {f"{name}__p{depth}": [float(x) for x in run["params"]]
                         for (name, depth), run in TRAINED.items()},
    "notes": "Exact-state optimization; finite shots only used for post-training validation. No hardware experiments."
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
for path in [optimization_path, sampling_path, manifest_path]:
    print("Saved:", path)